## Legal Chat Bot

### PART A: Divide our documents into chunks

In [1]:
import os
import glob
import tiktoken
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage

In [2]:
# price is a factor for testing, so we're going to use a low cost model
load_dotenv(override=True)

MODEL = "gpt-4.1-nano"
db_name = "chat_vector_db"
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")


OpenAI API Key exists and begins sk-or-v1


In [3]:
# How many characters in all the documents?
knowledge_base_path = "knowledge-base/*"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 7 files in the knowledge base
Total characters in knowledge base: 1,182,048


In [4]:
# How many tokens in all the documents?
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 219,752


In [5]:
# Load in everything in the knowledgebase using LangChain's loaders
documents = []
loader = DirectoryLoader("knowledge-base", glob="*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
folder_docs = loader.load()
for doc in folder_docs:
    documents.append(doc)    

print(f"Loaded {len(documents)} documents")

Loaded 7 documents


In [6]:
documents[1]

Document(metadata={'source': 'knowledge-base/Tenancy Disputes.md'}, page_content='# Tenancy Disputes and Legal Remedies in Nigerian Property Law: A Review of Judicial Decisions and Enforcement Challenges\n\n**Ikechukwu Chime, PhD**\nDepartment of Property Law\nFaculty of Law, University of Nigeria, Enugu Campus\nEmail: ike.chime@unn.edu.ng | ORCID: https://orcid.org/0009-1818-0410\n\n*REVIEW OF EDUCATION Vol. 33, Issue 2, 2021*\nhttp://instituteofeducation.unn.edu.ng/journal/\n\n---\n\n## Abstract\n\nThis paper examined tenancy disputes within the context of Nigerian property law, highlighting the statutory, common law, and policy dimensions that define the relationship between landlords and tenants. The study focused on the interpretation and enforcement of tenancy rights as regulated by the Rent Control and Recovery of Premises Acts and similar tenancy laws enacted by various states. It was established that disputes often arose from failure to pay rent, unlawful evictions, non-compli

In [7]:
# Divide into chunks using the RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,          # Larger chunks preserve more section context
    chunk_overlap=300,         # More overlap to avoid splitting mid-provision
    separators=[
        "\n## ",              # Major headings (Parts/Chapters)
        "\n### ",             # Sub-headings (Sections)
        "\n\n",               # Paragraph breaks
        "\n",                 # Line breaks
        ". ",                 # Sentences
        " ",                  # Words
    ]
)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 1101 chunks
First chunk:

page_content='# TENANCY LAW

**A LAW TO REGULATE RIGHTS AND OBLIGATIONS UNDER TENANCY AGREEMENTS AND THE RELATIONSHIP BETWEEN THE LANDLORD AND THE TENANT INCLUDING THE PROCEDURE FOR THE RECOVERY OF PREMISES IN LAGOS STATE AND FOR CONNECTED PURPOSES**

**[Commencement: 24th August 2011]**

The Lagos State House of Assembly enacts as follows—

---' metadata={'source': 'knowledge-base/Tenancy Law.md'}


## Add Metadata to Chunks
After splitting, enrich each chunk with source metadata:

In [8]:
for chunk in chunks:
    # Extract Act name and section from content or filename
    source_file = chunk.metadata.get("source", "")
    if "consumer" in source_file.lower():
        chunk.metadata["act"] = "FCCPA 2018"
    elif "1999" in source_file:
        chunk.metadata["act"] = "Constitution 1999"

### PART B: Make vectors and store in Chroma

Remember to set up a Hugging Face account and get your HF_TOKEN

Add it to your `.env` file and run `load_dotenv(override=True)`

(This actually shouldn't be required).

In [10]:
# Pick an embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

/Users/kelvini/Andela-LLM-Engineering/legal_ai_bot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10553.38it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 1101 documents


In [11]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 1,101 vectors with 384 dimensions in the vector store


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [12]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 8,                        # Retrieve more candidates
        "score_threshold": 0.3,         # Filter out low-relevance chunks
    }
)

llm = ChatOpenAI(temperature=0, model_name=MODEL, base_url="https://openrouter.ai/api/v1")

In [13]:
SYSTEM_PROMPT_TEMPLATE = """
You are a Nigerian legal information assistant. You help users understand their legal rights 
under Nigerian law by citing specific statutes, sections, and case precedents.

RULES:
1. ONLY cite laws, sections, and cases that appear in the provided context below.
2. Structure your response as: (a) applicable law, (b) relevant sections with quotes, 
   (c) practical next steps the user can take. (d) Expected outcome (e) Timeline
3. If the context does not contain relevant law for the question, say: 
   "The documents I have access to do not cover this area of law."
4. Always end with: "Disclaimer: This is legal information, not legal advice. 
   Please consult a qualified Nigerian lawyer for advice specific to your situation."
Keep it short, concise, and straight to the point.

Context:
{context}
"""

In [14]:
def expand_query(question: str) -> str:
    """Use LLM to add legal terms to the query for better retrieval."""
    expansion_prompt = f"""Given this user question about Nigerian law, 
    rewrite it to include relevant legal terminology that would help 
    find matching statutes. Keep it concise.
    
    Question: {question}
    Rewritten query:"""
    response = llm.invoke([HumanMessage(content=expansion_prompt)])
    return response.content

In [15]:
llm_question = expand_query("I bought a phone in Nigeria that broke after 1 week. Shop won't refund. What are my rights?")
llm_question

'What are my statutory rights under Nigerian consumer protection laws regarding warranty and refund obligations for a defective mobile phone purchased from a retailer?'

In [16]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [17]:
answer = answer_question(llm_question, [])
print(answer)

(a) Applicable law: Nigerian Consumer Protection Law, specifically sections 132 and 131.

(b) Relevant sections:
- **Section 132(1):** "In any transaction or agreement pertaining to the supply of goods to a consumer, there is an implied warranty that the goods shall comply with the requirements and standards contemplated in section 131(1) and (2) of this Act."
- **Section 132(2):** "Within three months after the delivery of any goods to a consumer, the consumer may return the goods to the undertaking that supplied those goods, without penalty and at the undertaking's risk and expense, if the goods fail to satisfy the requirements and standards contemplated in section 131(1) of this Act and the undertaking shall either repair or replace the failed, unsafe or defective goods or refund to the consumer the price paid by the consumer for the goods."

(c) Practical next steps:
- Notify the retailer within three months of purchase about the defect.
- Request a repair, replacement, or refund b